Importing corpus


In [ ]:
import os 
path=r"C:\Users\chaud\Downloads\imdb_text_dataset-20260817T104927Z-1-001\imdb_text_dataset"
filenames=os.listdir(path)
documents=[]
for filename in filenames:
    f=open(path+"//"+filename,'r')# whats double slash for, here making a file object. a pointer to the file. mode is r
    documents.append(f.read())
    f.close()

Regualar Expression Library For Normalization. 
1. Remove HTML tags.
2. Remove URLs
3. Remove email ids
4. To keep char only
5. remove extra whitespace

In [ ]:
import re
import nltk #regular expression
# *-zero or more occurrence, + means one or more occurrence, ? means no-greedy, # means search or optional char
# \s whitespace, \S- any char but whitespace(claude correct this. and organise this)
stopwords_list= nltk.corpus.stopwords.words('English')
from nltk.stem import PorterStemmer
def preprocess(text):
    text=text.lower()
    text=re.sub(r'<.+?',' ',text)  # < for html , . = any character, + = one or more occurrence, ?= Non greedy search (Claude explain this "?")
    text=re.sub(r'https?://\S+ | www\.\S+',' ',text)#?= prev character is optional, \S= not whitepace but anything. For dot we put slash.  
    text= re.sub(r'\S+@[a-zA-Z0-9]+\.[a-zA-Z]{2,}',' ',text) # curly braces for {min length, max length}
    text=re.sub(r'[^a-z]',' ',text )#  not ^ operator 
    text=re.sub(r'\s+',' ',text).strip() # strip for trailing spaces before and after 

    
    ps=PorterStemmer()
    text=[word for word in nltk.word_tokenize(text) if word not in stopwords_list]
    preprocessed_text=' '.join([ps.stem(word) for word in text])
    return preprocessed_text

In [ ]:
preprocessed_docs=[]
for doc in documents:
    preprocessed_docs.append(preprocess(doc))

In [ ]:
import pandas as pd
df=pd.DataFrame({"Reviews ": documents, "preprocessed_reviews": preprocessed_docs})

Binary TDM from scratch

In [ ]:
doc_count={}
for i in range (len(preprocessed_docs)):#for doc in preprocessed_docs:
    word_count={}
    for word in nltk.word_tokenize(preprocessed_docs[i]):
        if word not in word_count.keys():
            word_count[word]=1
        doc_count[filenames[i]]=word_count # dict of dict 
tdm_binary_scratch= pd.DataFrame(doc_count)
tdm_binary_scratch.fillna(0,inplace=True)

In [ ]:
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(binary=True)
X=cv.fit_transform(preprocessed_docs)
tdm_binary_inbuilt= pd.DataFrame(X.toarray().T,index=cv.get_feature_names(),columns=filenames)
tdm_binary_inbuilt

Term Frequency TDM. (FDM) Scratch


In [ ]:
doc_count1={}
for i in range (len(preprocessed_docs)):#for doc in preprocessed_docs:
    word_count1={}
    for word in nltk.word_tokenize(preprocessed_docs[i]):
        if word not in word_count1.keys():
            word_count1[word]=0
        word_count1[word]+=1
        doc_count1[filenames[i]]=word_count1 # dict of dict 
tdm_tf_scratch= pd.DataFrame(doc_count1)
tdm_tf_scratch.fillna(0,inplace=True)
tdm_tf_scratch=tdm_tf_scratch.sort_index()
tdm_tf_scratch

In [ ]:
#Revision - dont run 
document_count1={}
for i in range(len(preprocessed_docs)):
    word_count1={}
    for word in nltk.word_tokenize(preprocessed_docs[i]):
        if word not in word_count1.keys():
            word_count1[word]=0
        word_count1[word]+=1
    document_count1[filenames[i]]=word_count1
tdm_tf_scratch=pd.DataFrame(doc_count1)
tdm_tf_scratch.fillna(0,inplace=True)
tdm_tf_scratch=tdm_tf_scratch.sort_index()
tdm_tf_scratch

In [ ]:
tdm_len_scratch= tdm_tf_scratch.div(tdm_tf_scratch.sum(axis=0),axis=1) # take totals along rows and divide with column

In [ ]:
tdm_len_scratch.loc['one']

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv1=CountVectorizer()
y=cv1.fit_transform(preprocessed_docs)
tdm_tf_inbuilt= pd.DataFrame(y.toarray().T,index=cv1.get_feature_names_out,columns=filenames) # what is .T ??


In [ ]:
tdm_tf_inbuilt.loc['one']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer # see doc of this vectorizer
tf=TfidfVectorizer(use_idf=False, norm='l1',token_pattern= '(?u)\\b\\w+\\b')
Z=tf.fit_transform(preprocessed_docs)
tdm_len_inbuilt=pd.DataFrame(Z.toarray().T,index=tf.get_feature_names_out,columns=filenames)
tdm_len_inbuilt.loc['one']


Max norma

In [ ]:
tdm_max_scratch= tdm_tf_scratch.div(tdm_tf_scratch.max(axis=0),axis=1)
tdm_tf_scratch.max(axis=0)# max freq in each doc

In [ ]:
#log norma
import numpy as np

tdm_log_scratch = np.log(1+tdm_tf_scratch)

In [ ]:
tdm_log_scratch

Tf-idf yayayyyy

inverse document frequency  log ( total doc /no of docs its coming in)

In [ ]:
no_of_docs=tdm_tf_scratch.shape[1] # total no of docs , numerator
term_df= (tdm_tf_scratch>0).sum(axis=1)
idf=np.log(no_of_docs/term_df)

In [ ]:
idf

In [ ]:
tf_idf_scratch=tdm_tf_scratch.multiply(idf,axis=0)
tf_idf_scratch

idf inbuilt

In [ ]:
idf_scratch=1+np.log((no_of_docs+1)/(term_df+1)) # discuss aboout this in detail 
tf_idf=tdm_tf_scratch.multiply(idf_scratch,axis=0)
tf_idf
#l2 norm on the product of tf idf
l2_norm=np.sqrt((tf_idf**2).sum(axis=0))
tf_idf_scratch=tf_idf.div(l2_norm,axis=1)
tf_idf_scratch

Co-occurence matrix homework. binary tdm* 